# Expert Question Stats & Annotator Agreement

Number of expert questions per dataset, and pairwise annotator agreement (mean coverage) between experts.

**Source:** `data/eval/{dataset}/expert_*.yaml`, `outputs/annotator_agreement/{evaluator}/{dataset}/agreement.json`

In [10]:
import json
import re
from pathlib import Path

import pandas as pd
import yaml

_cwd = Path.cwd()
ROOT = _cwd if (_cwd / "pyproject.toml").exists() else _cwd.parent
DATA_EVAL_DIR = ROOT / "data" / "eval"
AGREEMENT_DIR = ROOT / "outputs" / "annotator_agreement"

CASE_NAMES = {
    "pl_age": "drug offences",
    "pl_medical_errors": "medical errors",
    "pl_personal_rights": "personal rights",
}


def fmt_label(name: str) -> str:
    return CASE_NAMES.get(name, re.sub(r"^[a-z]{2}_", "", name).replace("_", " "))

## Load data

In [11]:
question_records = []
for dataset_dir in sorted(DATA_EVAL_DIR.iterdir()):
    if not dataset_dir.is_dir():
        continue
    for expert_path in sorted(dataset_dir.glob("expert_*.yaml")):
        questions = yaml.safe_load(expert_path.read_text()).get("questions", [])
        question_records.append({
            "dataset": dataset_dir.name,
            "expert": expert_path.stem,
            "num_questions": len(questions),
        })
questions_df = pd.DataFrame(question_records)

agreement_records = []
for path in sorted(AGREEMENT_DIR.glob("**/agreement.json")):
    dataset = path.parent.name
    evaluator = path.parent.parent.name
    ad = json.loads(path.read_text())
    for pair in ad["pairs"]:
        t = pair["total_questions"]
        if t:
            agreement_records.append({
                "evaluator": evaluator,
                "dataset": dataset,
                "target": pair["target"],
                "reference": pair["reference"],
                "total_questions": t,
                "coverage": pair["covered_questions"] / t,
            })
agreement_df = pd.DataFrame(agreement_records)

print(
    f"{questions_df['dataset'].nunique()} dataset(s) · {len(questions_df)} expert file(s) · "
    f"{len(agreement_df)} annotator pairs"
)

3 dataset(s) · 9 expert file(s) · 18 annotator pairs


## Table — expert questions & annotator agreement by dataset

In [12]:
questions_summary = questions_df.groupby("dataset")["num_questions"].agg(["mean", "std"])
agreement_summary = agreement_df.groupby("dataset")["coverage"].agg(["mean", "std"])

summary = (
    questions_summary.join(agreement_summary, lsuffix="_q", rsuffix="_c")
    .rename(index=fmt_label)
    .rename_axis("Dataset")
)
summary.loc["Overall"] = [
    questions_df["num_questions"].mean(),
    questions_df["num_questions"].std(),
    agreement_df["coverage"].mean(),
    agreement_df["coverage"].std(),
]

table = pd.DataFrame({
    "Questions/expert": summary.apply(lambda r: f"{r['mean_q']:.1f} ± {r['std_q']:.1f}", axis=1),
    "Coverage": summary.apply(lambda r: f"{r['mean_c']:.1%} ± {r['std_c'] * 100:.1f}pp", axis=1),
})

display(table)


def latex_escape(s):
    return s.replace("%", "\\%").replace("±", "$\\pm$")


print(table.map(latex_escape).to_latex(column_format="l" + "c" * len(table.columns)))

,Questions/expert,Coverage
Dataset,,
drug offences,28.3 ± 9.6,56.4% ± 7.9pp
medical errors,33.0 ± 13.0,52.7% ± 19.0pp
personal rights,26.0 ± 3.0,52.3% ± 17.1pp
Overall,29.1 ± 8.8,53.8% ± 14.6pp


\begin{tabular}{lcc}
\toprule
 & Questions/expert & Coverage \\
Dataset &  &  \\
\midrule
drug offences & 28.3 $\pm$ 9.6 & 56.4\% $\pm$ 7.9pp \\
medical errors & 33.0 $\pm$ 13.0 & 52.7\% $\pm$ 19.0pp \\
personal rights & 26.0 $\pm$ 3.0 & 52.3\% $\pm$ 17.1pp \\
Overall & 29.1 $\pm$ 8.8 & 53.8\% $\pm$ 14.6pp \\
\bottomrule
\end{tabular}

